In [1]:
from dataclasses import dataclass, field
from pathlib import Path
import subprocess
import shutil
import psutil
import time
import os

current_path= Path(os.getcwd()).parent

In [ ]:
@dataclass
class DynaConfig:
    baseline_dir: Path = current_path / "baseline"
    bo_dir: Path = current_path / "bo-runs"
    lsdyna_exe: str = r"C:\LSDYNA\program\ls-dyna_smp_s_R16.0_619-gd222963e2f_winx64_ifort190_sse2.exe"
    input_file: str = "head.key"
    ncpu: int = 20
    memory: str = "1500m"
    timeout_hours: float = 2
    poll_interval: int = 10
    tests: list = field(default_factory=lambda: ["drop", "pendulum"])
    
def dyna_value(val):
    if isinstance(val, int):
        s = str(val)
        if len(s) > 10:
            s = f"{val:.3E}"
        return f"{s:<{10}}"
    abs_val = abs(val)
    if 1e-3 <= abs_val < 1e5:
        for decimals in range(6, -1, -1):
            s = f"{val:.{decimals}f}"
            if len(s) <= 10:
                return f"{s:<{10}}"
    for decimals in range(4, -1, -1):
        s = f"{val:.{decimals}E}"
        if len(s) <= 10:
            return f"{s:<{10}}"

def patch_params(params_path, parameters):
    with open(params_path, "r") as f:
        lines = f.readlines()
    for i in range(5,8):
        sect=lines[i].strip().split()
        dtype=sect[0]
        param_name=sect[1]
        if param_name in parameters:
                value = parameters[param_name]
                sect0 = f"{dtype:<1}"
                sect1 = f"{param_name:<8}"[:8]
                value_dyna = dyna_value(value)
                lines[i] = f"{sect0} {sect1}{value_dyna}\n"
    with open(params_path, "w") as f:
        f.writelines(lines)

def initialize_iters(
    run_id,
    parameters, cfg,
):
    baseline_dir=Path(cfg.baseline_dir)
    bo_dir=Path(cfg.bo_dir)
    bo_dir.mkdir(parents=True, exist_ok=True)
    run_dir = bo_dir / f"run{run_id}"

    if run_dir.exists():
        shutil.rmtree(run_dir)
    shutil.copytree(baseline_dir, run_dir)
    print(f"[SPAWNED] [{run_id}] {run_dir}")

    for test in cfg.tests:
        patch_params(run_dir/test/"params", parameters)
        print(f"[PATCHED] [{run_id}] [{test}] {run_dir/test/"params"}")

def dyna_exec(
    run_id,test,cfg
):
    test_path = Path(cfg.bo_dir) / Path("run"+str(run_id)) / test
    bat_path = test_path / "launchdyna.bat"
    bat_content = f"""@echo off
cd /d "{test_path}"
"{cfg.lsdyna_exe}" i={cfg.input_file} ncpu={cfg.ncpu} memory={cfg.memory}

"""
    with open(bat_path, "w") as f:
        f.write(bat_content)
    # print(f"[LAUNCH]  [{run_id}] [{test}] {bat_path}")
    return bat_path

def inspect_termination(test_path):
    d3hsp = Path(test_path) / "d3hsp"
    if not d3hsp.exists():
        print('[ERROR] d3hsp missing')
        return False
    try:
        with open(d3hsp, "r", errors="ignore") as f:
            lines = f.read()
        return "N o r m a l    t e r m i n a t i o n" in lines
    except Exception:
        return False

def killer(pid):
    parent = psutil.Process(pid)
    children = parent.children(recursive=True)
    for child in children:
        child.kill()
    parent.kill()

def launch_run(
    run_id, cfg
):
    bo_dir=Path(cfg.bo_dir)
    run_dir = bo_dir / f"run{run_id}"
    prcss = {}
    for test in cfg.tests:
        test_dir = run_dir/test
        bat_path = dyna_exec(run_id,test,cfg)
        print(f"[LAUNCH]  [{run_id}] [{test}] {test_dir}")
        p = subprocess.Popen(str(bat_path),cwd=test_dir,shell=True)
        prcss[test]=p
    start_time = time.time()
    try:
        while True:
            elapsed_hours = (time.time() - start_time) / 3600
            if elapsed_hours > cfg.timeout_hours:
                print(f"[TIMEOUT] [{run_id}] Exceeded {cfg.timeout_hours} hrs")
                for p in prcss.values():
                    killer(p.pid)
                raise TimeoutError(
                    f"[ERROR] Run {run_id} exceeded timeout"
                )
            done = all(p.poll() is not None for p in prcss.values())
            if done:
                checks = {
                    test: inspect_termination(run_dir / test)
                    for test in cfg.tests
                }
                if all(checks.values()):
                    print(f"[SOLVED]  [{run_id}] Normal termination for all models")
                    return True
                failed = [test for test, ok in checks.items() if not ok]
                raise RuntimeError(f"[FAILED] [{run_id}] {failed} execution failed")
            time.sleep(cfg.poll_interval)
    except KeyboardInterrupt:
        print("[INTERRUPT] Killing LS-DYNA")
        for p in prcss.values():
            killer(p.pid)
        raise

def blackbox(run_id, parameters, **overrides):
    cfg = DynaConfig(**overrides)
    initialize_iters(run_id=run_id, parameters=parameters, cfg=cfg)
    launch_run(run_id=run_id, cfg=cfg)


In [3]:
params = {
    "scale": 1.01,
    "rotate": 5,
    "slack": 5
}

blackbox(
    bo_dir=current_path/'temp',
    run_id=1, 
    parameters=params
)

[SPAWNED] [1] d:\Pranav\helmet-fit-bayes-opt\temp\run1
[PATCHED] [1] [drop] d:\Pranav\helmet-fit-bayes-opt\temp\run1\drop\params
[PATCHED] [1] [pendulum] d:\Pranav\helmet-fit-bayes-opt\temp\run1\pendulum\params
[LAUNCH]  [1] [drop] d:\Pranav\helmet-fit-bayes-opt\temp\run1\drop
[LAUNCH]  [1] [pendulum] d:\Pranav\helmet-fit-bayes-opt\temp\run1\pendulum
[SOLVED] [1] Normal termination for all models
